In [1]:
# CELL 1 — Dataset Download
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv",
    "spam.tsv"
)
print("✅ Dataset downloaded!")

✅ Dataset downloaded!


In [2]:
!pip install nltk scikit-learn pandas numpy -q
import nltk
nltk.download('stopwords')
print("✅ All libraries ready!")

✅ All libraries ready!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
import pandas as pd
import numpy as np
import re
import string
import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

df = pd.read_csv('spam.tsv', sep='\t', header=None, names=['label', 'message'])
print(f"Loaded {len(df)} messages | Spam: {(df['label']=='spam').sum()} | Ham: {(df['label']=='ham').sum()}")

stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

df['clean_message'] = df['message'].apply(preprocess)
print("Preprocessing done!")

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df['clean_message'])
y = (df['label'] == 'spam').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

models = {
    'Naive Bayes':         MultinomialNB(alpha=0.1),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVM (LinearSVC)':     LinearSVC(C=1.0, max_iter=2000),
}

print("\n" + "="*55)
print(f"{'Model':<22} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("="*55)

best_f1, best_model, best_name = 0, None, ""

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    print(f"  {name:<20} {acc:.4f}    {prec:.4f}    {rec:.4f}  {f1:.4f}")
    if f1 > best_f1:
        best_f1, best_model, best_name = f1, model, name

print("="*55)
print(f"\nBest Model: {best_name} (F1 = {best_f1:.4f})")
print(f"\nClassification Report — {best_name}")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

Loaded 5572 messages | Spam: 747 | Ham: 4825
Preprocessing done!
Train: 4457 | Test: 1115

Model                   Accuracy  Precision   Recall       F1
  Naive Bayes          0.9803    0.9847    0.8658  0.9214
  Logistic Regression  0.9641    1.0000    0.7315  0.8450
  SVM (LinearSVC)      0.9839    0.9712    0.9060  0.9375

Best Model: SVM (LinearSVC) (F1 = 0.9375)

Classification Report — SVM (LinearSVC)
              precision    recall  f1-score   support

         Ham       0.99      1.00      0.99       966
        Spam       0.97      0.91      0.94       149

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [4]:
def predict_spam(message):
    cleaned = preprocess(message)
    features = vectorizer.transform([cleaned])
    pred = best_model.predict(features)[0]
    label = "SPAM 🚨" if pred == 1 else "HAM ✅"
    score = best_model.decision_function(features)[0]
    conf = 1 / (1 + np.exp(-abs(score))) * 100
    return label, conf

tests = [
    "Congratulations! You've won a FREE iPhone. Click here now!!!",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT: Your account is suspended. Call now.",
    "Can you send me today's notes?",
    "Win Rs 50,000 cash prize! Call 9999 now to claim!",
]

for msg in tests:
    label, conf = predict_spam(msg)
    print(f"{label}  ({conf:.1f}%)  →  {msg[:50]}")

HAM ✅  (50.6%)  →  Congratulations! You've won a FREE iPhone. Click h
HAM ✅  (82.5%)  →  Hey, are we still meeting for lunch tomorrow?
SPAM 🚨  (53.9%)  →  URGENT: Your account is suspended. Call now.
HAM ✅  (63.2%)  →  Can you send me today's notes?
SPAM 🚨  (75.2%)  →  Win Rs 50,000 cash prize! Call 9999 now to claim!
